# ⚖️ 04. 에이전트 평가 체계 & LLM-as-a-Judge (Offline Evaluation · 메트릭 설계 · 프롬프트 자동 튜닝)

1~3번 노트북에서 구축한 `main_agent`는 몇 가지 예시 질문에는 자연스럽게 답합니다. 하지만 프로덕션 환경에 배포하려면 **어떤 유형의 질문에서 오답·환각·복구 실패가 나거나 불필요한 도구 호출이 반복되는지 정량 수치로 확인하고, 모델이 아니라 에이전트 하네스(도구 인터페이스 · 프롬프트 · 미들웨어 · 컨텍스트) 중 어디를 고쳐야 하는지 진단하는 과정**이 필요합니다.

이번 노트북에서는 에이전트 엔지니어링의 **EDD(Evaluation-Driven Development)** 핵심 흐름(오프라인 벤치마크 데이터셋 · 커스텀 메트릭 · 계층별 실패 진단 → 오프라인 프롬프트 자동 튜닝)을 실습합니다.

---

### 🎯 핵심 학습 목표
1. **골든 데이터셋 & 시드(`EvalSeed`) 기반 문항 생성·검수**로 `main_agent`의 핵심 역량을 정량 평가하는 기준선을 세웁니다.
2. **기본 메트릭(`app.evaluate.metrics`)**과 **커스텀 메트릭(2단계 LLM 체인 + 비용 0원 Python 코드 검증기)**을 조합해 통합 결과 테이블과 하네스 계층별 진단 표를 산출합니다.
3. **배포 전 배치 평가(Offline Evaluation)**와 **실행 중 자가 교정(`Runtime Self-Correction`)**의 역할·한계(정답 라벨 유무, 지연/스트리밍 UX 제약)를 구분하고, CLI 회귀 게이트 및 `pass@k`/`pass^k` 신뢰도 측정 방법을 익힙니다.
4. **오프라인 프롬프트 자동 튜닝(Train/Val 분리 힐 클라이밍 + 진화 차트/Diff 시각화 + DSPy `MIPROv2`/`COPRO`/`BootstrapFewShot` 심화 아키텍처)**을 실습합니다.

### 📚 학습 목차

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 프로젝트 루트, 에이전트 모델(`gemini-3.8-flash`), 타 계열 판사 모델(`gpt-5.4-mini`) 초기화 |
| **Part 1** | 배포 전 평가 vs. 런타임 자가 교정 & 진단 계층 | Offline Evaluation과 Runtime Self-Correction 비교, `main_agent` 하네스 4계층 진단 기준 |
| **Part 2** | 골든 데이터셋 & 시드 기반 생성·검수 | 10문항 `main_agent` 벤치마크 구조, `EvalSeed` 기반 문항 생성 및 판사 검수 |
| **Part 3** | 기본 메트릭 + 커스텀 메트릭 조합 | `grounding_2step`(LLM) · `artifact_and_evidence`(Python 코드) 작성 및 `EvalMetric` 리스트 결합 |
| **Part 4** | `main_agent` 베이스라인 벤치마크 & 진단 | 10문항 병렬 채점(`results_table`), 실패 계층 진단(`diagnosis_table`), 비용 산점도 및 CLI 회귀 안내 |
| **Part 5** | 오프라인 프롬프트 자동 튜닝 (Hill Climbing & DSPy) | Train/Val 분리 + 길이 페널티 최적화, 진화 과정 차트/Diff 시각화, DSPy 심화 아키텍처 |
| **Part 6** | 정리 · FAQ · 심화 과제 | 전체 요약, 실무 FAQ, `pass@k`/`pass^k` 심화 과제 및 샌드박스 정리 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 탐색해 `sys.path`에 등록하고 `.env` 환경변수를 로드합니다.  
채점 시 같은 모델이 자기 답변을 후하게 채점하는 **자기 선호 편향(Self-Preference Bias)**을 줄이기 위해, **평가 대상 에이전트(`gemini-3.8-flash`)** 와 **심사위원 판사 모델(`gpt-5.4-mini`)** 을 서로 다른 모델 계열로 분리해 초기화합니다.

In [ ]:
import os
import sys
import warnings
import nest_asyncio
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# 1. 프로젝트 루트 경로 상향 탐색 (어느 하위 폴더에서 실행해도 동작)
project_root = os.path.abspath(os.getcwd())
while project_root != os.path.dirname(project_root):
    if os.path.exists(os.path.join(project_root, "app")) and (
        os.path.exists(os.path.join(project_root, ".env")) or os.path.exists(os.path.join(project_root, "configs"))
    ):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 및 Jupyter 비동기 루프 설정
load_dotenv(os.path.join(project_root, ".env"), override=True)
nest_asyncio.apply()

demo_dir = os.path.join(project_root, "artifacts", "eval_sandbox")
os.makedirs(demo_dir, exist_ok=True)

# 3. 평가 대상 LLM(Gemini)과 타 계열 Judge LLM(OpenAI) 초기화
from app.utils import init_chat_model, normalize_content
from app.evaluate import JUDGE_MODEL

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)
judge_llm = init_chat_model(model=JUDGE_MODEL, temperature=0.0)

print(f"✅ Working Directory : {os.getcwd()}")
print(f"✅ Sandbox Directory : {demo_dir}")
print(f"🤖 Agent LLM: gemini-3.8-flash | ⚖️ Judge LLM: {JUDGE_MODEL}")

---
## Part 1. 왜 평가가 필요한가: 배포 전 평가 vs. 런타임 자가 교정

### 1.1 눈으로 보는 데모의 한계와 평가·검증 방식의 역할 분담

에이전트에 질문 두세 개를 던져 보는 것만으로는 운영 현장의 숨은 결함을 잡기 어렵습니다. 특히 10~20여 개 도구를 조율하는 `main_agent`에서는 다음 세 가지 문제가 자주 발생합니다.

- **조용한 오답 (Silent Failure)**: 파일을 읽고 수식을 계산할 때 조건 필터링 하나를 빠뜨리거나 단위를 오해해도, 겉보기엔 아주 매끄러운 한국어 보고서를 출력합니다.
- **과잉 탐색으로 인한 지연·비용 폭증**: 존재하지 않는 데이터를 물었을 때 최종 답변은 *"찾을 수 없다"*고 맞히더라도, 그 결론을 내기 전까지 `glob_search`와 `grep_search`를 불필요하게 연쇄 호출해 토큰과 응답 시간을 낭비합니다.
- **장애 복구 실패**: 손상된 파일을 만났을 때 대체 파일(백업본)을 찾지 못하거나, `enter_plan` / `exit_plan` 같은 하네스 규약을 어깁니다.

이 결함을 다루는 방식은 **정답 라벨(Ground Truth) 유무**와 **응답 지연 제약**에 따라 크게 **배포 전 배치 평가(Offline Evaluation)**와 **실행 중 자가 교정(Runtime Self-Correction)**, 그리고 **오프라인 프롬프트 자동 튜닝**으로 나누어 설계합니다.

| 구분 | 배포 전 배치 평가 (`Offline Evaluation`, Part 2~4) | 런타임 자가 교정 (`Runtime Self-Correction` · 개념 비교) | 오프라인 프롬프트 자동 튜닝 (`Hill Climbing` / `DSPy`, Part 5) |
|:---|:---|:---|:---|
| **실행 시점** | PR 생성, 도구·프롬프트·모델 변경 시 배치 실행 | 사용자 요청 처리 도중, **최종 답변 반환 직전** | 오프라인 개발/튜닝 배치 단계 |
| **판정 기준 · 대상** | **정답 라벨·루브릭** 기반 사실 정확도·궤적·회귀(Regression) 검증 | 정답 라벨 없이 **질문·도구 결과·답변 간 정합성** 대조 및 재작성 | 시스템 프롬프트 지시문(`Proposer` + `Validation Set`) 최적화 |
| **지연시간 제약** | 여유 있음 (병렬 배치 실행, 다회차 `pass^k` 측정 가능, 사용자 지연 **0ms**) | 엄격함 (추가 지연 1~3초 및 스트리밍 UX를 고려해 정확도 우선 작업에 선택 적용) | 오프라인 배치 튜닝 (사용자 지연 **0ms**) |
| **핵심 가치** | 실패 계층 진단 및 배포 전 회귀 차단 | 누락·수치 불일치 발견 시 내부 피드백으로 단건 응답 보완 | 실패 트레이스 기반 지시문의 점진적·객관적 최적화 |

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 18, 'nodeSpacing': 30, 'rankSpacing': 46}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    G["1. 골든 데이터셋<br/>(유형별 질문 · 정답 · 도구)"] --> R["2. main_agent 실행<br/>(답변 · 궤적 · 산출물)"]
    R --> S["3. 정량 채점<br/>(기본 + 커스텀 메트릭)"]
    S --> D["4. 하네스 계층 진단<br/>(도구 ACI · 프롬프트 · 미들웨어)"]
    D --> F["5. 하네스 · 프롬프트 개선<br/>(도구 신호 · 지시문 자동 튜닝)"]
    F --> RE(["6. 전체 문항 회귀 재평가"])
    RE --> G
```

### 1.2 `main_agent` 하네스 4계층 구조와 원인 진단 기준

문제가 생길 때마다 시스템 프롬프트에 예외 규칙을 한 줄씩 덧붙이는 방식은 프롬프트 비대화(Prompt Bloating)와 지침 충돌을 부릅니다.  
**좋은 평가 하네스는 "점수가 낮다"에서 멈추지 않고, `main_agent` 하네스의 4계층 중 어디를 고쳐야 하는지 정확히 가리켜야 합니다.**

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 18, 'nodeSpacing': 28, 'rankSpacing': 52}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    subgraph L1["1계층: 도구 인터페이스 (ACI)"]
        T1["파일 · 검색 · 쉘 도구<br/>(file_read, glob_search, grep_search, bash_command)"]
        T2["계획 보드 도구<br/>(enter_plan, exit_plan, task_create)"]
    end
    subgraph L2["2계층: 컨텍스트 · 스킬 · 메모리"]
        C1["PromptAssembler (Static + User Context)"]
        C2["SkillPromptBuilder (SKILL.md 카탈로그)"]
    end
    subgraph L3["3계층: 실행 제어 · 복구 미들웨어"]
        M1["ToolErrorHandler & ModelCallLimit"]
        M2["StopHooks & EvaluatorHarness"]
    end
    subgraph L4["4계층: 프롬프트 & 평가 기준"]
        P1["SUPERVISOR_SYSTEM_PROMPT"]
        P2["golden_eval_dataset.json"]
    end
    L1 --> L3
    L2 --> L4
```

| 점검 계층 | 주로 발생하는 결함 예시 | 이 레포지토리의 수정 위치 |
|:---|:---|:---|
| **1계층: 도구 인터페이스 (ACI)** | 없는 파일을 읽을 때 주변 파일 목록을 안 알려줘 불필요한 `glob`/`grep` 연쇄를 유발함 | `app/tools/common.py`, `app/tools/plan.py` |
| **2계층: 컨텍스트 · 스킬** | 스킬 카탈로그나 `SKILL.md` 안내가 불명확해 스크립트 경로/인자를 잘못 호출함 | `app/agents/main_agent/skills/*/SKILL.md`, `app/middleware/prompt/` |
| **3계층: 실행 제어 · 복구** | 도구 에러 시 우회하지 못하고 멈추거나, 미들웨어 피드백이 턴 경계를 오염시킴 | `app/middleware/error_control/`, `app/middleware/evaluator/`, `app/utils/message_markers.py` |
| **4계층: 프롬프트 · 평가셋 라벨** | 탐색 중단 조건이 모호하거나, 정상 대체 경로(`A \| B`)가 평가 라벨에 빠져 억울하게 감점됨 | `app/agents/main_agent/prompt.py`, `data/eval/golden_eval_dataset.json` |

---
## Part 2. 평가 데이터셋 구성과 시드(Seed) 기반 생성·검수

### 2.1 `main_agent` 골든 데이터셋 (`golden_eval_dataset.json`) 살펴보기

`data/eval/golden_eval_dataset.json`에는 `main_agent`의 핵심 역량(로컬 데이터 분석, 수식 계산, 코드베이스 탐색, 설정 점검, 손상 파일 복구, 파일 산출물 생성, 계획 모드 규약, Progressive Skills, 미존재 데이터 방어)을 검증하는 **10개 평가 문항**이 들어 있습니다.

- **`ground_truth` (기준 정답)**: 최종 답변에 포함되어야 할 핵심 수치, 파일·심볼명, 계산 근거, 미존재 결론 기준
- **`target_tools` (기대 호출 도구)**: 과업 해결에 필요한 권장 도구 목록 (대체 가능한 도구는 `"file_read | grep_search"`처럼 `|`로 표기해 정상 우회 경로를 불이익 없이 인정)
- **`expected_artifact` (선택)**: `art-01`처럼 디스크에 실제 파일을 생성해야 하는 문항은 기대 파일 경로를 함께 명시해 **코드 검증기가 파일 내용을 직접 파싱**합니다.

In [ ]:
from IPython.display import display
from app.evaluate import load_golden

golden = load_golden()
golden_df = pd.DataFrame(golden)[["id", "category", "question", "ground_truth", "target_tools"]]
golden_df.columns = ["문항 ID", "유형", "질문", "기준 정답 (ground_truth)", "기대 호출 도구 (target_tools)"]
display(golden_df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))

위 10개 문항이 점검하려는 주요 상황을 요약하면 다음과 같습니다.

| 평가 유형 | 해당 문항 | 점검하려는 핵심 역량 |
|:---|:---|:---|
| 파일 조회 · 다중 조건 · 수식 계산 | `data-01`, `calc-01`, `cafe-01` | JSON 필터링, 다른 파일의 수식 규칙 적용, 가중 평균 정확도 |
| 코드베이스 · 설정 파일 탐색 | `code-01`, `cfg-01` | Python 모듈 심볼 분석(`message_markers.py`), `configs/*.config` 다중 파일 종합 |
| 손상 파일 장애 복구 | `err-01` | `.bin` 손상 파일 오류 인지 후 동일 폴더 백업 JSON(`backup_laptops_catalog.json`)으로 자동 복구 |
| 파일 산출물 생성 (State Verifier) | `art-01` | 말로만 "저장했다"고 하지 않고 실제로 정렬된 JSON 파일을 디스크에 정확히 기록하는지 검증 |
| 계획 모드 · Progressive Skills | `plan-01`, `skill-01` | `enter_plan`→조사→`exit_plan`→`task_create` 순서 준수, `SKILL.md` 동적 열람 |
| 존재하지 않는 데이터 (과잉 탐색 방지) | `neg-01` | 없는 스마트폰 가격을 지어내지 않고, 불필요한 전수 검색 헛돌기 없이 탐색을 종료하는지 확인 |

### 2.2 시드(Seed) 기반 평가 데이터셋 생성과 타 계열 판사 자동 검수

초기 서비스 배포 전 사용자 로그가 없을 때 파일 전체를 LLM에 넣고 *"문제 10개 만들어줘"*라고 시키면, 파일 경로가 빠지거나 기준 정답이 모호한 저품질 문항이 섞이기 쉽습니다.  
실무에서는 엔지니어가 **시드(`EvalSeed`: 평가 유형 · 주제 · 시나리오 · 원천 파일 · 기대 도구)** 를 먼저 정의하고, 생성된 초안을 **타 계열 판사 모델로 3가지 기준(`is_grounded`, `is_standalone`, `is_clear_rubric`)** 을 검수해 통과한 문항만 채택합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 18, 'nodeSpacing': 32, 'rankSpacing': 48}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    S["1. EvalSeed 정의<br/>(유형 · 시나리오 · 원천 파일)"] --> G["2. 문항 초안 생성<br/>(generate_from_seeds)"]
    G --> V{"3. 판사 모델 검수<br/>(근거 · 독립성 · 명확성)"}
    V --> H["4. 검수 통과 → 담당자 확인"]
    V --> X["검수 미달 → 시드 보정"]
    H --> D["5. golden_eval_dataset.json 추가"]
```

In [ ]:
from app.evaluate import EvalSeed, generate_from_seeds, verify_generated_cases

seeds = [
    EvalSeed(
        id="syn-01",
        category="설정 파일 검증",
        topic="메모리 및 트랜스크립트 DB 설정 확인",
        scenario="configs/memory.config 파일을 읽어 semantic_memory와 episodic_memory 활성화 여부, top_k_prefetch 값, transcript.db 경로를 정확히 확인하는 질문",
        source_files=["configs/memory.config"],
        target_tools=["file_read"],
    ),
    EvalSeed(
        id="syn-02",
        category="가중치 정책 분석",
        topic="카페 상권 평가 가중치 및 추천 기준 확인",
        scenario="artifacts/notebooks/cafe_data/scoring_weights.json을 읽어 가장 비중이 큰 상위 2개 지표와 최소 통과 점수(minimum_score_threshold)를 묻는 질문",
        source_files=["artifacts/notebooks/cafe_data/scoring_weights.json"],
        target_tools=["file_read"],
    ),
    EvalSeed(
        id="syn-03",
        category="존재하지 않는 데이터",
        topic="카페 상권 데이터에 없는 배달 매출 지표 문의",
        scenario="artifacts/notebooks/cafe_data/districts.json에 실제로 없는 '배달 앱 월 매출액' 컬럼을 묻고, 해당 항목이 데이터에 없어 확인되지 않음을 안내해야 하는 질문",
        source_files=["artifacts/notebooks/cafe_data/districts.json"],
        target_tools=["file_read"],
    ),
]

draft_cases = await generate_from_seeds(seeds, llm=llm)
verified_df, accepted_cases = await verify_generated_cases(draft_cases, seeds, judge_llm=judge_llm)

display(verified_df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))
print(f"\n검수 통과: {len(accepted_cases)}/{len(draft_cases)}건 -> 담당자 최종 확인 후 golden_eval_dataset.json에 편입 가능")

---
## Part 3. 기본 메트릭과 커스텀 메트릭 조합 (`EvalMetric`)

### 3.1 기본 메트릭과 커스텀 메트릭: 어떻게 나누어 설계할까?

모든 평가 지표를 매번 새로 짤 필요는 없습니다. 실무 평가 하네스는 **검증된 기본 메트릭**과 **도메인 규칙에 맞춘 커스텀 메트릭**을 `EvalMetric` 인터페이스로 묶어 하나의 통합 테이블에서 비교합니다.

| 구분 | 지표 예시 | 측정 내용 | 채점 구현 방식 |
|:---|:---|:---|:---|
| **기본 제공 메트릭**<br/>(`app.evaluate.metrics`) | `task_completion` (과업 완결)<br/>`tool_selection` (도구 선택)<br/>`step_economy` (호출 효율)<br/>`error_recovery` (장애 복구) | 기준 정답과 핵심 사실 대조, 적합한 도구 선택, 불필요한 반복/과잉 탐색 억제, 도구 에러 우회 복구 | CoT-First 구조화 판사(`HarnessJudge`)가 문항당 궤적/결과를 1회씩 캐시 채점 |
| **커스텀 LLM 메트릭**<br/>(직접 정의) | `grounding_2step`<br/>(2단계 근거 충실도) | 답변을 단일 사실 주장(Claim)으로 분해한 뒤 도구 관측 결과(`contexts`)만 보고 환각 검증 | 문장 분해 → 질문 배제 NLI 대조 **2단계 LLM 체인** |
| **커스텀 Python 코드 메트릭**<br/>(직접 정의 · 비용 0원) | `artifact_and_evidence`<br/>(산출물 상태 & 근거 인용) | `expected_artifact` 지정 시 디스크 JSON 파일을 직접 파싱 검증, 일반 문항은 출처 파일/심볼 인용 검사 | **결정론적 Python 함수** (API 비용 0원, 지연 0ms) |

### 3.2 [실습 1] 커스텀 LLM 메트릭: 2단계 근거 충실도 (`score_grounding_2step`)

단일 프롬프트로 *"근거 충실도를 0~1로 채점해줘"*라고 요청하면 긴 답변 속에 숨은 1줄의 지어낸 수치를 놓치거나 점수가 흔들립니다.  
이를 해결하기 위해 검증 체인을 **2단계**로 분해합니다.

1. **1단계 — 단일 사실 주장 분해 (`ExtractedClaims`)**: 답변에서 검증 가능한 짧은 사실 문장(`claims`)을 추출합니다. 이때 *"파일/데이터가 확인되지 않습니다"* 같은 정당한 미존재 안내(`is_abstention=True`)를 별도로 표시해 주장이 0개일 때 0점으로 오판되는 함정을 막습니다.
2. **2단계 — 질문 제외 문맥 NLI 대조 (`ClaimVerdict`)**: 판사 모델에게 **사용자 질문을 빼고 도구 실행 결과(`contexts`)와 주장(`claims`)만 전달**해 각 문장이 관측 결과로 뒷받침되는지(`yes` / `no` / `idk`) 판정합니다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class ExtractedClaims(BaseModel):
    is_abstention: bool = Field(
        description="답변의 핵심 결론이 '요청한 파일/정보가 존재하지 않거나 확인되지 않는다'는 정당한 미존재 안내인지 여부"
    )
    claims: list[str] = Field(
        description="답변에서 추출한 독립적인 단일 사실 주장 목록 (인사말·형식적 안내문 제외)"
    )


class ClaimVerdict(BaseModel):
    claim: str = Field(description="검증 대상 단일 사실 주장")
    evidence: str = Field(description="도구 관측 결과 내 관련 근거 또는 불일치/부재 사유 (1문장)")
    verdict: Literal["yes", "no", "idk"] = Field(
        description=(
            "관측 결과로 뒷받침·계산 가능하면 'yes'; "
            "관측과 모순되거나 관측 결과에 없는 수치·보증·사은품·혜택·스펙을 사실처럼 단정한 주장은 반드시 'no'(절대 'idk'로 분류 금지); "
            "단순 연결 문장 또는 정당한 미존재 안내만 'idk'"
        )
    )


class GroundingReport(BaseModel):
    verdicts: list[ClaimVerdict]


CLAIM_EXTRACT_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "주어진 [에이전트 답변]을 검증 가능한 단일 사실 주장 단위로 분해하세요.\n"
        "1. 대명사('해당 파일', '그 모델')는 문맥상 원래 이름으로 풀어 적으세요.\n"
        "2. 복합문은 단일 사실 단위로 쪼개고 인사말·맺음말은 제외하세요.\n"
        "3. 결론이 '해당 데이터/파일이 확인되지 않는다'이고 없는 수치를 지어내지 않았다면 is_abstention=True로 표시하세요.",
    ),
    ("human", "[에이전트 답변]\n{answer}"),
])

# 2단계에는 사용자 질문을 빼고 도구 관측 결과(contexts)와 주장(claims)만 전달합니다.
NLI_VERIFY_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 사실 검증 감사관입니다. 오직 [도구 관측 결과]만을 근거로 각 주장을 판정하세요.\n"
        "- 사전 지식이나 일반 상식을 근거로 허용하지 마세요.\n"
        "- 'yes': [도구 관측 결과]에 직접 명시되어 있거나, 관측 수치로부터 정확히 산술 계산된 사실\n"
        "- 'no': [도구 관측 결과]와 모순되거나, 관측 결과에 전혀 언급되지 않은 수치·보증·사은품·혜택·스펙을 사실처럼 단정한 주장 "
        "(관측에 없는 구체적 속성·조건·혜택 주장은 절대 'idk'가 아니라 반드시 'no'로 판정)\n"
        "- 'idk': 오직 '~가 존재하지 않는다 / 확인되지 않는다' 같은 정당한 미존재 안내 문장이나 사실 단정이 없는 단순 연결 문장에만 제한적으로 사용",
    ),
    ("human", "[도구 관측 결과]\n{contexts}\n\n[검증할 주장 목록]\n{claims}"),
])


async def score_grounding_2step(answer: str, contexts: list[str]) -> dict:
    extracted = await (CLAIM_EXTRACT_PROMPT | judge_llm.with_structured_output(ExtractedClaims)).ainvoke(
        {"answer": answer}
    )
    if extracted.is_abstention and not extracted.claims:
        return {"score": 1.0, "is_abstention": True, "verdicts": []}
    if not extracted.claims:
        return {"score": 0.0, "is_abstention": False, "verdicts": []}

    context_text = "\n---\n".join(contexts) if contexts else "[도구 관측 결과 없음]"
    numbered = "\n".join(f"{i+1}. {c}" for i, c in enumerate(extracted.claims))
    report = await (NLI_VERIFY_PROMPT | judge_llm.with_structured_output(GroundingReport)).ainvoke(
        {"contexts": context_text, "claims": numbered}
    )
    yes_cnt = sum(v.verdict == "yes" for v in report.verdicts)
    no_cnt = sum(v.verdict == "no" for v in report.verdicts)
    denom = yes_cnt + no_cnt
    score = 1.0 if (denom == 0 and extracted.is_abstention) else (yes_cnt / denom if denom else 0.0)
    return {"score": round(score, 2), "is_abstention": extracted.is_abstention, "verdicts": report.verdicts}


# 3종 샘플 즉시 동작 테스트
sample_ctx = [
    '[File: laptops_market_data.json] {"id": "LP-002", "brand": "LG전자", "model": "2026 그램 프로 16", "price_krw": 1890000, "weight_kg": 1.19, "rating": 4.9}'
]
samples = [
    ("정상 근거 답변", "LG전자 2026 그램 프로 16(LP-002)의 무게는 1.19kg, 평점은 4.9점, 가격은 1,890,000원입니다."),
    ("교묘한 환각 혼입", "LG전자 2026 그램 프로 16의 가격은 1,890,000원이며, 무상 보증 기간은 3년이고 사은품 마우스를 증정합니다."),
    ("정당한 미존재 안내", "제공된 관측 결과(laptops_market_data.json)에는 노트북 정보만 확인되며, 요청하신 스마트폰 가격 정보는 존재하지 않습니다."),
]

for title, ans in samples:
    res = await score_grounding_2step(ans, sample_ctx)
    print(f"[{title}] Grounding = {res['score']:.2f} (미존재 보호={res['is_abstention']})")
    for v in res["verdicts"]:
        print(f"   - [{v.verdict:<3}] {v.claim} <- {v.evidence}")

### 3.3 [실습 2] 커스텀 Python 코드 메트릭 (비용 0원): 산출물 상태 & 근거 인용 검증 (`score_artifact_and_evidence`)

파일이 실제로 저장되었는지, JSON 형식이 유효하고 정렬되어 있는지, 답변에 근거 파일·심볼·식별자가 명시되었는지는 LLM 판사에게 물을 필요가 없습니다.  
**코드로 확정 판정할 수 있는 항목은 Python 함수로 작성**하면 API 비용이 **0원**이고 실행 시간도 **1ms 미만**입니다.

- `expected_artifact`가 지정된 문항(`art-01`): 디스크 경로의 실제 JSON 파일을 열어 배열 구조, 내림차순 정렬, 1위 상권(`홍대입구`) 여부를 **결정론적 State Verifier**로 검사합니다.
- 일반 문항: 답변에 근거 파일/모듈/설정/스킬/식별자 인용이 있거나 정당한 미존재 안내 표현이 포함되어 있는지 정규식으로 검사합니다.

In [ ]:
import json
import re

EVIDENCE_PATTERN = re.compile(
    r"([\w./-]+\.(?:json|py|md|bin|config)|LP-0\d|ORIGIN_[A-Z_]+|is_user_turn_start|pdf_extractor|pdf2text|enter_plan|exit_plan|task_create|성수동|망원동|홍대입구|여의도|삼성역)"
)
ABSTENTION_PATTERN = re.compile(r"(확인되지\s*않|존재하지\s*않|없습니다|찾을\s*수\s*없)")


def score_artifact_and_evidence(answer: str, expected_artifact: str = "") -> dict:
    '''산출물 경로가 지정되면 디스크 파일 상태를 직접 파싱 검증하고, 아니면 답변의 근거 파일/심볼 인용을 검사합니다.'''
    if expected_artifact:
        if not os.path.exists(expected_artifact):
            return {"score": 0.0, "check": "artifact_missing", "path": expected_artifact}
        try:
            with open(expected_artifact, "r", encoding="utf-8") as f:
                data = json.load(f)
            if not isinstance(data, list) or len(data) < 5:
                return {"score": 0.4, "check": "invalid_row_count", "rows": len(data) if isinstance(data, list) else 0}
            effs = [float(item.get("efficiency", 0)) for item in data if isinstance(item, dict) and "efficiency" in item]
            is_sorted = len(effs) == len(data) and effs == sorted(effs, reverse=True)
            top_district = data[0].get("district") if isinstance(data[0], dict) else ""
            if is_sorted and top_district == "홍대입구":
                return {"score": 1.0, "check": "artifact_verified", "top": top_district}
            return {"score": 0.5, "check": "order_or_top_mismatch", "top": top_district, "sorted": is_sorted}
        except Exception as e:
            return {"score": 0.0, "check": f"parse_error: {e}"}

    refs = sorted(set(EVIDENCE_PATTERN.findall(answer or "")))
    is_abstention = bool(ABSTENTION_PATTERN.search(answer or ""))
    return {
        "score": 1.0 if (refs or is_abstention) else 0.0,
        "references": refs[:5],
        "is_abstention": is_abstention,
    }


print("근거 인용 답변 :", score_artifact_and_evidence("laptops_market_data.json 기준 LP-002 모델 가격은 1,890,000원입니다."))
print("근거 누락 답변 :", score_artifact_and_evidence("가격은 1,890,000원입니다."))
print("미존재 안내 답변:", score_artifact_and_evidence("요청하신 파일은 존재하지 않습니다."))

### 3.4 [실습 3] 기본 메트릭과 커스텀 메트릭을 하나의 리스트로 조합하기

작성한 커스텀 함수 2종(`score_grounding_2step`, `score_artifact_and_evidence`)을 `EvalMetric`으로 감싸고, `app.evaluate.metrics`의 기본 메트릭 4종(`task_completion`, `tool_selection`, `step_economy`, `error_recovery`)과 하나의 리스트(`benchmark_metrics`)로 결합합니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 18, 'nodeSpacing': 30, 'rankSpacing': 52}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    subgraph P["기본 제공 메트릭 (app.evaluate.metrics)"]
        P1["과업 완결 (0.30) · 도구 선택 (0.15)<br/>호출 효율 (0.15) · 장애 복구 (0.10)"]
    end
    subgraph C["커스텀 메트릭"]
        C1["근거 충실 2단계 LLM (0.20)<br/>산출물·인용 Python 코드 (0.10)"]
    end
    P1 --> L["benchmark_metrics 리스트"]
    C1 --> L
    L --> E["evaluate_agent() 병렬 실행"]
    E --> T["results_table() 통합 표"]
```

In [ ]:
from app.evaluate import (
    EvalMetric,
    error_recovery,
    metrics_table,
    step_economy,
    task_completion,
    tool_selection,
)

grounding_2step = EvalMetric(
    name="grounding_2step",
    label="근거 충실(2단계)",
    fn=score_grounding_2step,
    weight=0.20,
    threshold=0.7,
    diagnosis="근거 없는 생성",
    suspect_layer="답변 원칙, 근거 외 추측 억제",
)

artifact_and_evidence = EvalMetric(
    name="artifact_evidence",
    label="산출물·인용",
    fn=score_artifact_and_evidence,
    weight=0.10,
    threshold=0.5,
    diagnosis="산출물/근거 누락",
    suspect_layer="파일 저장 검증, 출처 표기 지침",
)

benchmark_metrics = [
    task_completion,
    tool_selection,
    step_economy,
    error_recovery,
    grounding_2step,
    artifact_and_evidence,
]

display(metrics_table(benchmark_metrics).style.hide(axis="index"))

---
## Part 4. `main_agent` 베이스라인 벤치마크 실행 및 통합 결과 표

`build_main_agent_for_eval()`로 실제 `main_agent` 하네스(`SUPERVISOR_SYSTEM_PROMPT`, `PromptAssembler`, `SkillPromptBuilder`, `ToolErrorHandlerMiddleware`, `ModelCallLimitMiddleware`, `StopHooksMiddleware` + 로컬 코어 도구 11종)를 조립해 10개 골든 문항을 병렬 평가합니다.

> 💡 **동시 실행 격리(`plan_session_scope` + `eval_sandbox`)**:  
> `concurrency=4`로 병렬 평가할 때 `plan-01`의 계획 보드와 `art-01`의 출력 파일이 서로 섞이지 않도록 `run_case`가 문항마다 고유 `thread_id`와 격리된 `eval_sandbox/{thread_id}` 디렉토리를 자동 할당합니다.

In [ ]:
from app.evaluate import build_main_agent_for_eval
from tests.agent_inspector import extract_agent_middlewares, extract_agent_tools

agent = build_main_agent_for_eval(model=llm)
print("🛠️ 바인딩된 도구:", [t.name for t in extract_agent_tools(agent)])
print("🧩 마운트된 미들웨어:", [type(m).__name__ for m in extract_agent_middlewares(agent)])

In [ ]:
from app.evaluate import diagnosis_table, evaluate_agent, results_table, summary_line

print("🚀 베이스라인 벤치마크 실행 중 (10문항, 기본 4종 + 커스텀 2종 / 약 1~2분 소요)\n")
rows = await evaluate_agent(
    agent,
    golden,
    metrics=benchmark_metrics,
    judge_llm=judge_llm,
    concurrency=4,
    sandbox_root=demo_dir,
)

print("\n[요약] " + summary_line(rows))
print("\n[문항별 통합 채점 결과 표 — 기본 메트릭 + 커스텀 메트릭]")
display(results_table(rows).style.format(precision=2).hide(axis="index"))

print("[진단 기준 미달 항목 및 의심 하네스 계층 집계]")
display(diagnosis_table(rows).style.hide(axis="index"))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["font.family"] = "DejaVu Sans"

df = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
colors = ["#2f855a" if p else "#c53030" for p in df["passed"]]
ax.scatter(df["tool_calls"], df["composite"], s=95, c=colors, alpha=0.85, edgecolors="white")
for _, r in df.iterrows():
    ax.annotate(r["id"], (r["tool_calls"], r["composite"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axhline(0.75, color="#e53e3e", linestyle="--", linewidth=1.2, label="Pass Threshold (0.75)")
ax.set_xlabel("Tool Calls (count)")
ax.set_ylabel("Composite Score")
ax.set_title("Baseline Benchmark (main_agent): Composite Score vs. Tool Calls")
ax.set_ylim(0, 1.08)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

### 통합 결과 테이블과 산점도 읽는 법

- **기본 지표와 커스텀 지표의 교차 검증**: `results_table(rows)`에서 `과업 완결`, `도구 선택`, `호출 효율`, `장애 복구` 옆에 커스텀 메트릭인 `근거 충실(2단계)`와 `산출물·인용` 점수가 나란히 표시됩니다.
- **최종 점수만 보면 놓치기 쉬운 비용 병목**: 산점도에서 오른쪽에 치우친 문항은 최종 답변을 맞게 내더라도, 결론에 도달하기 전까지 도구 호출 수와 토큰을 많이 소비하는 지점입니다.
- **하네스 수정 후 회귀 검증 (CLI 게이트)**: 진단 표(`diagnosis_table`)를 바탕으로 도구 반환 형식(ACI)이나 프롬프트를 고친 뒤에는 `app/evaluate/runner.py` CLI로 기준 리포트 대비 회귀(허용 오차 초과 점수 하락)를 자동 점검할 수 있습니다.
  ```bash
  # 기준 리포트 생성
  python -m app.evaluate.runner --concurrency 4
  # 수정 후 이전 리포트와 비교 + 허용 오차 초과 하락 시 exit code 1 반환
  python -m app.evaluate.runner --compare artifacts/eval_runs/main_agent_xxxx.json --fail-on-regression
  ```
- **비결정성 정량화 (`pass@k` vs `pass^k`)**: `evaluate_agent(..., trials=3)` 또는 `--trials 3` 옵션을 주면 한 문항을 3회 반복 실행해 **잠재 역량(`pass@3`: 3번 중 1번 이상 성공)** 과 **프로덕션 신뢰도(`pass^3`: 3번 모두 연속 성공)** 차이를 함께 측정할 수 있습니다.

---
## Part 5. 오프라인 프롬프트 자동 튜닝 (Hill Climbing · 시각화 · DSPy 아키텍처)

### 5.1 런타임 자가 교정 vs. 오프라인 프롬프트 자동 튜닝

| 구분 | 런타임 자가 교정 (`Runtime Self-Correction`) | 오프라인 프롬프트 자동 튜닝 (`Hill Climbing` / `DSPy`, Part 5) |
|:---|:---|:---|
| **최적화 대상** | 개별 요청의 최종 답변(Final Answer) | 시스템 프롬프트(Instruction) · Few-shot 예시 자체 |
| **실행 시점** | 실행 중 (사용자 요청 응답 반환 직전) | 오프라인 개발/튜닝 배치 단계 |
| **과적합 · 비대화 방지** | 단건 기준 충족 또는 상한 도달 시 즉시 종료 | **Train/Validation 엄격 분리** + **길이 페널티(Length Penalty)** |
| **핵심 메커니즘** | Judge → 내부 피드백 → 답변 재작성 | `Proposer` 변형 → `Validation` 채점 → Accept/Reject |

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 18, 'nodeSpacing': 30, 'rankSpacing': 46}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    P0["1. Current Prompt"] --> PR["2. Proposer LLM<br/>(Train 실패 트레이스 분석 · 변형 제안)"]
    PR --> CAND["3. 변형 프롬프트 후보"]
    CAND --> VAL["4. Validation Set 채점<br/>+ Length Penalty 목적함수"]
    VAL -->|"개선됨 (Accept)"| P0
    VAL -.->|"개선 없음 (Reject · 폐기)"| PR
```

In [ ]:
# 5.2 [실습] Train/Validation 분리 + 길이 페널티 기반 힐 클라이밍 프롬프트 자동 튜닝
from app.evaluate import hill_climbing_optimize

train_set = [
    {
        "id": "tr-01",
        "query": "LP-001 모델의 Value_Score를 공식에 따라 계산 과정과 함께 보고하세요.",
        "tool_context": "specs_criteria: Value_Score=(RAM_GB*10+SSD_GB/10+Rating*20)/(Price_KRW/100000) | LP-001: RAM=16, SSD=512, Rating=4.7, Price=599000",
        "requirements": "계산 과정(분자 305.2, 분모 5.99)과 최종 점수 50.95점, 근거 파일명 명시",
    },
    {
        "id": "tr-02",
        "query": "성수동·망원동 후보지와 설문 가중 평균 객단가를 보고하세요.",
        "tool_context": "districts: 성수동(boost=1.6, rent=220000), 망원동(boost=1.3, rent=150000) | survey: 42%@6500, 28%@12000, 18%@5500, 12%@4800",
        "requirements": "후보지 2곳의 수치와 가중 평균 객단가 7,656원 계산 근거 명시",
    },
]

val_set = [
    {
        "id": "val-01",
        "query": "smartphones_market_data.json 파일 확인 결과를 보고하세요.",
        "tool_context": "[TARGET NOT FOUND] Directory inventory: backup_laptops_catalog.json, corrupted_catalog.bin, laptops_market_data.json, specs_criteria.json",
        "requirements": "파일 미존재 안내 및 디렉토리 내 실제 파일 목록 인용, 없는 가격 추측 금지",
    },
    {
        "id": "val-02",
        "query": "홍대입구와 성수동의 유동인구 대비 임대료 효율을 비교해 1위를 보고하세요.",
        "tool_context": "홍대입구: foot_traffic=85000, rent=280000 | 성수동: foot_traffic=62000, rent=220000",
        "requirements": "각 상권 효율 수치(0.3036 vs 0.2818) 계산 근거와 1위(홍대입구) 명시",
    },
]

baseline_prompt = "사용자의 질문에 핵심 결론만 최대한 짧게 한 문장으로 답하세요."
optimized_prompt, climb_history = hill_climbing_optimize(
    baseline=baseline_prompt,
    dataset=train_set,
    val_dataset=val_set,
    model=llm,
    judge_llm=judge_llm,
    proposer_llm=judge_llm,
    length_penalty_alpha=0.05,
    max_iterations=2,
    target_score=95.0,
    verbose=True,
)

### 5.3 프롬프트 진화 과정 시각화 & Before/After Unified Diff 분석

프롬프트 튜닝도 감으로 고치는 것이 아니라 **이터레이션별 검증 점수(`avg_score`), 길이 페널티를 반영한 목적함수(`objective_score`), 채택 여부(`accepted`), 그리고 줄 단위 변경 내역(`Unified Diff`)** 으로 추적해야 합니다.

In [ ]:
import difflib
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.family"] = "DejaVu Sans"

iterations = [h["iteration"] for h in climb_history]
val_scores_hist = [h["avg_score"] for h in climb_history]
obj_scores_hist = [h.get("objective_score", h["avg_score"]) for h in climb_history]
bar_colors = ["#2ecc71" if h["accepted"] else "#e74c3c" for h in climb_history]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (좌) 반복별 Validation 점수 막대 (Accepted=초록 / Rejected=빨강) + 목적함수 추이
axes[0].bar(iterations, val_scores_hist, color=bar_colors, alpha=0.85, edgecolor="white", label="Val Score")
axes[0].plot(iterations, obj_scores_hist, "o--", color="#2c3e50", linewidth=2, markersize=7, label="Objective (Length Penalty)")
axes[0].set_xticks(iterations)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Score")
axes[0].set_title("Hill Climbing: Validation vs. Objective Score")
axes[0].set_ylim(0, 112)
for x, val_y, obj_y in zip(iterations, val_scores_hist, obj_scores_hist):
    axes[0].annotate(f"{val_y:.1f}\n(obj {obj_y:.1f})", (x, val_y), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=9)
axes[0].legend(loc="lower right", fontsize=9)

# (우) 채택된 최고 점수 궤적
best_scores = []
running_best = 0.0
for h in climb_history:
    if h["accepted"]:
        running_best = h["avg_score"]
    best_scores.append(running_best)

axes[1].fill_between(iterations, best_scores, alpha=0.25, color="#3498db")
axes[1].plot(iterations, best_scores, "s-", color="#2980b9", linewidth=2.2, markersize=7, label="Best Accepted Val Score")
axes[1].set_xticks(iterations)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Best Score")
axes[1].set_title("Best Accepted Score Trajectory")
axes[1].set_ylim(0, 112)
axes[1].legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

print("\n📜 [Before vs After 프롬프트 Unified Diff]")
full_diff = list(difflib.unified_diff(
    baseline_prompt.splitlines(),
    optimized_prompt.splitlines(),
    fromfile="Before (Baseline)",
    tofile="After (Optimized)",
    lineterm="",
))
print("\n".join(full_diff) if full_diff else "(변경 없음)")

### 5.4 [심화] 현업 수준의 프롬프트 자동 튜닝 아키텍처 (Beyond Vanilla Hill Climbing — DSPy 중심)

실습에서 구현한 Hill Climbing(Train 실패 트레이스 → Validation 채택 + 길이 페널티)은 핵심 원리를 이해하기에 가장 투명한 구조입니다.  
실제 상용 LLMOps 환경에서는 여기에 **Stanford DSPy** 같은 선언적 프롬프트 최적화 프레임워크의 4가지 기법이 결합됩니다.

#### 1. 탐색 알고리즘 고도화: Greedy → Beam Search (`DSPy COPRO` / `MIPROv2`) · Evolutionary (`GEPA` / *Promptbreeder*)
- **Greedy Hill Climbing의 한계**: 매 턴 단일 후보만 평가하므로 한 번 잘못된 방향으로 변형되면 **지역 최적점(Local Optima)**에 갇히기 쉽습니다.
- **실무 해결책**: 매 턴 `K개`(예: 3~5개)의 변형 지시문을 동시에 생성·평가하고 상위 `B개` 후보만 유지하는 **Beam Search(`DSPy COPRO`)**, 또는 실행 궤적 반성(Reflective Mutation)과 Pareto 병합을 수행하는 **진화 알고리즘(`GEPA` · DeepMind *Promptbreeder*)**을 사용합니다.

#### 2. 데이터셋 3분할 격리: Train / Validation / Test
- **Train Set**: Proposer가 실패 원인을 분석해 지시문 변형·데모를 생성할 때만 사용
- **Validation Set**: 후보 프롬프트의 채택(Accept/Reject) 및 하이퍼파라미터 선택에 사용 (이번 실습 5.2 적용)
- **Test Set**: 최종 선발된 프롬프트의 일반화 성능을 배포 직전 단 1회 블라인드 검증

#### 3. 목적 함수의 길이 페널티(Length Penalty)
- 프롬프트에 예외 규칙이 무한정 붙으면 토큰 비용 증가와 Attention 희석(*Lost in the Middle*)이 일어납니다. 이번 실습에서 적용한 **`목적함수 = 검증 점수 - alpha × 프롬프트 문자 수`** 페널티로 간결성과 정확도의 균형을 맞춥니다.

#### 4. 지시문 튜닝을 넘어 Few-shot Demonstrations 자동 채굴 (`DSPy BootstrapFewShot` · `MIPROv2`)
- **Instruction-only 튜닝의 한계**: 긴 시스템 프롬프트를 고치는 것보다 **실제 평가셋에서 성공한 최적 궤적(Trajectory) 2~4건을 자동으로 추려 Few-shot 예시(Demonstrations)로 주입**할 때 성능과 형식 준수율이 훨씬 크게 상승합니다.
- **DSPy 선언적 컴파일 개념 패턴** *(참고용 개념 스니펫 — 별도 `dspy` 패키지 설치 시 적용 가능)*:
```python
import dspy

class GroundedAgentSignature(dspy.Signature):
    '''도구 관측 결과만을 근거로 정확하고 간결하게 보고합니다.'''
    tool_context: str = dspy.InputField(desc="도구 실행 결과 및 디렉토리 인벤토리")
    question: str = dspy.InputField(desc="사용자 요청")
    answer: str = dspy.OutputField(desc="근거 파일명과 계산 과정이 포함된 최종 답변")

# eval_metric(example, pred, trace=None)은 Part 3의 EvalMetric / Judge 점수를 그대로 연결 가능
optimizer = dspy.MIPROv2(metric=agent_eval_metric, auto="medium")
compiled_program = optimizer.compile(
    dspy.ChainOfThought(GroundedAgentSignature),
    trainset=dspy_train,
    valset=dspy_val,
)
```

#### 5. 실무 프롬프트 자동 최적화 생태계 비교

| 프레임워크 / 엔진 | 핵심 최적화 전략 | 강점 및 적합 상황 |
|:---|:---|:---|
| **Stanford DSPy (`COPRO`, `BootstrapFewShot`, `MIPROv2`)** | **지시문(Instruction) + Few-shot 예시**를 동시에 베이지안/트레이스 기반으로 컴파일 | 모듈형 파이프라인 전체의 프롬프트를 데이터 기반으로 재컴파일할 때 표준 |
| **DSPy `GEPA` / DeepMind *Promptbreeder*** | 실행 트레이스 반성(Reflective Mutation) + 다중 후보 교차(Evolutionary) | 복잡한 멀티스텝 에이전트의 도구 사용 규칙과 프롬프트를 함께 진화시킬 때 적합 |
| **Google DeepMind OPRO** | 과거 후보와 점수 이력(Meta-Prompt)을 LLM에 제시해 점진 개선 | 추론·수식형 태스크의 지시문 최적화 |
| **Anthropic Prompt Optimizer / LangSmith Promptim** | 실패 케이스·평가자 피드백을 반영해 콘솔/CI에서 자동 PR 제안 | 기존 운영 트레이스·평가셋이 이미 구축된 서비스의 지속적 개선 |

---
## Part 6. 정리 · FAQ · 심화 과제

| 단계 | 이번 노트북에서 구축·검증한 내용 | 실무 적용 포인트 |
|:---|:---|:---|
| **1. 데이터셋 & 시드 (Part 2)** | `main_agent` 10문항 벤치마크 + `EvalSeed` 시드 생성·타 계열 판사 검수 | 배포 전엔 시드 생성+검수로 초안을 확보하고, 도메인별 대체 경로(`A \| B`)를 라벨링 |
| **2. 조합형 메트릭 (Part 3)** | `PRESET_METRICS` 4종 + `score_grounding_2step`(2단계 LLM) + `score_artifact_and_evidence`(비용 0원 코드) | 파일·포맷·산출물 상태 검증은 Python 함수로, 의미/환각 검증은 질문 배제 2단계 체인으로 분리 |
| **3. 벤치마크 & 진단 (Part 4)** | `evaluate_agent` 병렬 실행 → `results_table` · `diagnosis_table` · 산점도로 실패 계층 파악 | 점수가 낮을 때 프롬프트부터 수정하기보다 도구(ACI)·컨텍스트·미들웨어 중 어느 계층인지 먼저 분리 |
| **4. 프롬프트 자동 튜닝 (Part 5)** | Train/Val 분리 + 길이 페널티 힐 클라이밍, 차트/Diff 시각화, DSPy(`MIPROv2`/`BootstrapFewShot`) 아키텍처 | 하네스 결함을 먼저 정리한 뒤 남는 지시문 격차를 Train/Val 분리 + 길이 페널티 + Few-shot 채굴로 마무리 |

### ❓ 실무에서 자주 묻는 질문 (FAQ)

- **Q1. 왜 Judge 모델(`gpt-5.4-mini`)을 에이전트 모델(`gemini-3.8-flash`)과 다른 계열로 두나요?**  
  같은 모델 계열이 자신이 생성한 문체와 추론 패턴을 채점할 때 나타나는 **자기 선호 편향(Self-Preference Bias)**을 줄이고, 판정의 독립성을 높이기 위해서입니다.
- **Q2. 오프라인 평가(Part 4)와 런타임 자가 교정은 어떻게 나누어 쓰나요?**  
  배포 전 **Offline Evaluation**은 정답 라벨을 기준으로 사실 정확도와 회귀(`Regression`)를 막는 용도입니다. 반면 **런타임 자가 교정**은 정답 라벨 없이 질문·도구 결과·답변 사이의 정합성만 확인하며 응답 지연(1~3초)이 생기므로, 일반 스트리밍 대화보다 보고서·코드 패치처럼 정확도가 중요한 작업에 선택적으로 적용합니다.
- **Q3. 런타임 자가 교정과 프롬프트 자동 튜닝(Part 5)의 차이는 무엇인가요?**  
  런타임 자가 교정은 **특정 1건의 답변**을 반환 직전에 고쳐 쓰는 장치이고, 프롬프트 자동 튜닝은 여러 실패 트레이스를 바탕으로 **시스템 프롬프트 자체**를 오프라인에서 개선해 다음 실행부터 전반적인 성공률을 높이는 과정입니다.

### 🚀 더 해 보기 (심화 실험 아이디어)
1. **비결정성 신뢰도 측정 (`trials=3`)**: `await evaluate_agent(agent, golden[:4], metrics=benchmark_metrics, judge_llm=judge_llm, trials=3)`를 실행해 `pass@3`과 `pass^3`의 격차를 확인해 보세요.
2. **커스텀 메트릭 확장**: `plan-01`이나 `code-01` 같은 문항에 대해 필수 심볼·단계 호출 순서를 코드로 검사하는 새 `EvalMetric`을 작성해 `benchmark_metrics`에 추가해 보세요.
3. **CI 회귀 게이트 실행**: 터미널에서 아래 명령으로 CLI 리포트 비교와 종료 코드 게이트를 직접 테스트해 보세요:
   ```bash
   python -m app.evaluate.runner --ids data-01 err-01 neg-01 --trials 1
   ```

In [ ]:
# 실습용 임시 샌드박스 및 eval_runner 세션 아티팩트 정리
import shutil

for target in (
    demo_dir,
    os.path.join(project_root, "artifacts", "users", "eval_runner"),
    os.path.join(project_root, "artifacts", "users", "demo_user"),
):
    if os.path.exists(target):
        shutil.rmtree(target, ignore_errors=True)
        print(f"🧹 정리 완료: {target}")